# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library. All dataset elements are referenced by their `@id` as specified by the Croissant schema.

### Dataset Source
The dataset is described via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as a single object
metadata = dataset.metadata

# Print basic dataset info
name = metadata.name
description = metadata.description
print(f"{name}: {description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All record set references use their `@id` per Croissant specification.

In [ ]:
# Explore available record sets in the dataset
record_sets = list(dataset.record_sets())
print("Available Record Sets:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '(no name)')}")

# For demonstration, show fields in the first record set
if record_sets:
    record_set_id = record_sets[0]['@id']
    print(f"\nFields in Record Set `{record_set_id}`:")
    fields = dataset.fields(record_set=record_set_id)
    for field in fields:
        print(f"  - Field @id: {field['@id']}, name: {field.get('name', '')}, type: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the `@id`s identified above.

In [ ]:
# Prepare to load all record sets into DataFrames
dataframes = {}
rs_ids = [rs['@id'] for rs in record_sets]

for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {rs_id}, shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))

# Choose the first record set for EDA
primary_record_set_id = rs_ids[0] if rs_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All operations reference fields by their `@id`.

In [ ]:
# If no record set loaded, skip EDA
if primary_record_set_id is not None and not dataframes[primary_record_set_id].empty:
    df = dataframes[primary_record_set_id]

    # Identify numeric fields (@id of numeric columns)
    # We'll scan columns with 'age' or similar - you may want to review fields above
    numeric_candidates = [col for col in df.columns if 'age' in col.lower() or df[col].dtype in [int, float]]

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field for filtering: {numeric_field_id}")

        threshold = 40  # Example threshold for age
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Group by categorical field (@id)
        # Try to find 'sex', 'msi_status', 'location', etc. as group candidates
        group_candidates = [col for col in df.columns if any(x in col.lower() for x in ['sex', 'msi', 'location'])]
        group_field_id = group_candidates[0] if group_candidates else None
        if group_field_id:
            print(f"Grouping records by {group_field_id}:")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
    else:
        print("No obvious numeric field found for EDA.")
else:
    print("No tabular record sets loaded. Please check the schema.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All visualizations reference columns by their Croissant `@id`.

In [ ]:
# Plotting distributions using field @id
if primary_record_set_id is not None and not dataframes[primary_record_set_id].empty and numeric_candidates:
    df = dataframes[primary_record_set_id]
    numeric_field_id = numeric_candidates[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} values")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id found, visualize group means
    if group_field_id:
        grouped_df = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated end-to-end FAIR data exploration using `mlcroissant`:
- All entities were referenced by their Croissant `@id` field
- Record sets and fields were dynamically discovered and loaded
- Exploratory filtering, normalization, and grouping were performed referencing IDs
- Data distributions and relationships were visualized

These steps enable reproducible, standards-based data analysis and processing of clinical FAIR^2 datasets.